# PatchCore self-test on MVTec AD (`capsule`)
Standalone Colab run of the training-free **PatchCore** pipeline from
**industrial-anomaly-detection**. Downloads the capsule category (standard MVTec
folder layout) from a Hugging Face mirror, fits on defect-free images only, and
reports real image/pixel AUROC + a localization heatmap.
**Runtime → T4 GPU**, then **Runtime → Run all**.

In [ ]:
# 1) GPU check + download the capsule category (MVTec layout) from a HF mirror
import os, subprocess, time, torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
URL = "https://huggingface.co/datasets/alexsu52/mvtec_capsule/resolve/main/capsule.tar.xz"
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/capsule"):
    t = time.time()
    subprocess.run(["wget", "-q", "-O", "capsule.tar.xz", URL], check=True)
    subprocess.run(["tar", "-xf", "capsule.tar.xz", "-C", "data"], check=True)
    print("downloaded + extracted in %.0fs" % (time.time() - t))
print("capsule subfolders:", sorted(os.listdir("data/capsule")))

In [ ]:
# 2) PatchCore (WideResNet50, layer2+layer3) -> fit, evaluate, heatmaps
import torch, torch.nn.functional as F, numpy as np, glob, os, time
from PIL import Image
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt, matplotlib.cm as cm

torch.manual_seed(42); np.random.seed(42)
dev = "cuda" if torch.cuda.is_available() else "cpu"
ROOT, CAT, SIZE = "data", "capsule", 224
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
itf = transforms.Compose([transforms.Resize(256, InterpolationMode.BILINEAR),
                          transforms.CenterCrop(SIZE), transforms.ToTensor(),
                          transforms.Normalize(mean, std)])
mtf = transforms.Compose([transforms.Resize(256, InterpolationMode.NEAREST),
                          transforms.CenterCrop(SIZE), transforms.ToTensor()])

def find_mask(gt, dt, stem):
    for cand in (f"{gt}/{dt}/{stem}_mask.png", f"{gt}/{dt}/{stem}.png"):
        if os.path.exists(cand):
            return cand
    g = glob.glob(f"{gt}/{dt}/{stem}*.png")
    return g[0] if g else None

def load_split(split):
    imgs, labels, masks = [], [], []
    base, gt = f"{ROOT}/{CAT}/{split}", f"{ROOT}/{CAT}/ground_truth"
    for d in sorted(glob.glob(base + "/*")):
        dt = os.path.basename(d); good = dt == "good"
        for p in sorted(glob.glob(d + "/*.png")):
            imgs.append(itf(Image.open(p).convert("RGB")))
            labels.append(0 if good else 1)
            if good:
                masks.append(torch.zeros(1, SIZE, SIZE))
            else:
                mp = find_mask(gt, dt, os.path.splitext(os.path.basename(p))[0])
                masks.append((mtf(Image.open(mp).convert("L")) > 0.5).float()
                             if mp else torch.zeros(1, SIZE, SIZE))
    return torch.stack(imgs), torch.tensor(labels), torch.stack(masks)

Xtr, ytr, _ = load_split("train")
Xte, yte, Mte = load_split("test")
print("train", tuple(Xtr.shape), "| test", tuple(Xte.shape), "| test defects", int(yte.sum()))

net = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1).to(dev).eval()
for p in net.parameters(): p.requires_grad_(False)
feat = {}
net.layer2.register_forward_hook(lambda m, i, o: feat.__setitem__("l2", o))
net.layer3.register_forward_hook(lambda m, i, o: feat.__setitem__("l3", o))

@torch.no_grad()
def patch_feats(x):
    feat.clear(); net(x.to(dev))
    hw = feat["l2"].shape[-2:]; parts = []
    for k in ["l2", "l3"]:
        f = F.avg_pool2d(feat[k], 3, 1, 1)
        parts.append(F.interpolate(f, size=hw, mode="bilinear", align_corners=False))
    e = torch.cat(parts, 1); b, c, h, w = e.shape
    return e.permute(0, 2, 3, 1).reshape(b, h, w, c), (h, w)

def gauss(maps, sigma=4.0):
    r = max(int(4 * sigma + 0.5), 1); xx = torch.arange(-r, r + 1.0, device=maps.device)
    k = torch.exp(-0.5 * (xx / sigma) ** 2); k /= k.sum(); m = maps.unsqueeze(1)
    m = F.conv2d(F.pad(m, (r, r, 0, 0), mode="reflect"), k.view(1, 1, 1, -1))
    m = F.conv2d(F.pad(m, (0, 0, r, r), mode="reflect"), k.view(1, 1, -1, 1))
    return m.squeeze(1)

t = time.time(); bank = []
for i in range(0, len(Xtr), 16):
    pf, hw = patch_feats(Xtr[i:i + 16]); bank.append(pf.reshape(-1, pf.shape[-1]))
bank = torch.cat(bank, 0); H, W = hw

def kcenter(Fm, ratio=0.01, pdim=128, seed=42):
    n = Fm.shape[0]; k = max(1, int(n * ratio))
    g = torch.Generator(device="cpu").manual_seed(seed)
    proj = (torch.randn(Fm.shape[1], pdim, generator=g) / pdim ** 0.5).to(Fm.device, Fm.dtype)
    P = Fm @ proj; s = int(torch.randint(0, n, (1,), generator=g))
    sel = [s]; md_ = torch.cdist(P, P[s:s + 1]).squeeze(1)
    for _ in range(1, k):
        j = int(md_.argmax()); sel.append(j)
        md_ = torch.minimum(md_, torch.cdist(P, P[j:j + 1]).squeeze(1))
    return torch.tensor(sel, device=Fm.device)

mem = bank[kcenter(bank)]
print(f"fit {time.time() - t:.0f}s | patches {bank.shape[0]} -> memory {mem.shape[0]} (1%) | grid {H}x{W}")

def nn_min(q, b, chunk=8192):
    return torch.cat([torch.cdist(q[s:s + chunk], b).min(1).values
                      for s in range(0, q.shape[0], chunk)], 0)

@torch.no_grad()
def predict(x):
    pf, _ = patch_feats(x); b = pf.shape[0]; q = pf.reshape(-1, pf.shape[-1])
    d = nn_min(q, mem).reshape(b, H, W)
    qf = q.reshape(b, H * W, -1); ps = d.reshape(b, H * W); scores = []
    for i in range(b):
        p = int(ps[i].argmax()); xs = qf[i, p:p + 1]
        im = int(torch.cdist(xs, mem).argmin()); ms = mem[im:im + 1]
        nb = mem[torch.topk(torch.cdist(ms, mem).squeeze(0), 9, largest=False).indices]
        dxn = torch.cdist(xs, nb).squeeze(0)
        scores.append((1 - torch.softmax(dxn, 0)[0]) * ps[i, p])
    smap = F.interpolate(d.unsqueeze(1), size=(SIZE, SIZE), mode="bilinear",
                         align_corners=False).squeeze(1)
    return torch.stack(scores).cpu(), gauss(smap).cpu()

t = time.time(); all_s, all_m = [], []
for i in range(0, len(Xte), 16):
    s, m = predict(Xte[i:i + 16]); all_s.append(s); all_m.append(m)
S = torch.cat(all_s).numpy(); MP = torch.cat(all_m).numpy(); secs = time.time() - t
img_auroc = roc_auc_score(yte.numpy(), S)
pix_auroc = roc_auc_score((Mte.squeeze(1).numpy().reshape(-1) > 0.5).astype(int), MP.reshape(-1))
print("\n==== RESULTS (capsule, PatchCore WRN50, 1% coreset) ====")
print(f"image AUROC = {img_auroc:.4f}")
print(f"pixel AUROC = {pix_auroc:.4f}")
print(f"inference   = {1000 * secs / len(Xte):.1f} ms/image over {len(Xte)} images")

def denorm(t): return np.clip(t.numpy().transpose(1, 2, 0) * std + mean, 0, 1)
idx = [i for i, l in enumerate(yte.tolist()) if l == 1][:4]
fig, ax = plt.subplots(len(idx), 4, figsize=(12, 3 * len(idx)))
for r, i in enumerate(idx):
    a = MP[i]; a = (a - a.min()) / (a.max() - a.min() + 1e-9); base = denorm(Xte[i])
    ov = np.clip(0.5 * base + 0.5 * cm.jet(a)[..., :3], 0, 1)
    panels = [(base, None, "input"), (Mte[i, 0].numpy(), "gray", "ground truth"),
              (a, "jet", "heatmap"), (ov, None, "overlay")]
    for c, (im, cmap, ttl) in enumerate(panels):
        ax[r, c].imshow(im, cmap=cmap); ax[r, c].axis("off")
        if r == 0: ax[r, c].set_title(ttl)
plt.tight_layout(); plt.savefig("capsule_patchcore.png", dpi=140, bbox_inches="tight"); plt.show()
print("saved heatmap -> capsule_patchcore.png")